In [1]:
import os
import numpy as np
import pandas as pd
import pyedflib
from pyedflib import highlevel
from scipy.signal import resample
from sklearn.utils import shuffle
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
TARGET_SAMPLE_RATE = 200  # Hz
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

MAX_TIMEPOINT_PER_TRIAL = 30000000  # limit the maximum number of timepoints per trial to avoid overfitting
MIN_TIMEPOINT_PER_TRIAL = 20000    # minimum number of timepoints per trial to be included

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200   # e.g. 0 means 0% overlap for SEG_LEN=400 (This dataset is large enough, no need for overlap)

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
# majority session sampling rate: 256Hz, both downsampling & upsampling are needed
# remember to delete the eeg data with 1Hz sampling frequency, but standard 19 channels do not have the abnormal frequencies.

# root path of epilepsy & no epilepsy edf files
root_path = 'TUEP//v2.0.0'
path_ep = 'TUEP//v2.0.0//epilepsy_edf'
path_noep = 'TUEP//v2.0.0//no_epilepsy_edf'

# 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
li_std_ch_1 = ['EEG FP1-REF','EEG FP2-REF','EEG F7-REF','EEG F3-REF','EEG FZ-REF','EEG F4-REF','EEG F8-REF','EEG T3-REF','EEG C3-REF',
            'EEG CZ-REF','EEG C4-REF','EEG T4-REF','EEG T5-REF','EEG P3-REF','EEG PZ-REF','EEG P4-REF','EEG T6-REF','EEG O1-REF','EEG O2-REF']
li_std_ch_2 = ['EEG FP1-LE','EEG FP2-LE','EEG F7-LE','EEG F3-LE','EEG FZ-LE','EEG F4-LE','EEG F8-LE','EEG T3-LE','EEG C3-LE',
            'EEG CZ-LE','EEG C4-LE','EEG T4-LE','EEG T5-LE','EEG P3-LE','EEG PZ-LE','EEG P4-LE','EEG T6-LE','EEG O1-LE','EEG O2-LE']

In [5]:
# select subjects all 19 standard channels in li_std_ch
def std_sub(root, li_std_ch_1, li_std_ch_2, min_num=2, max_num=40):
    li_std_sub = []
    unique_channels = set()
    for file in os.listdir(root):
        p = os.path.join(root, file)
        if os.path.isdir(p):
            edf_path = os.path.join(root, file)
            for sub in os.listdir(edf_path):
                sub_path = os.path.join(edf_path, sub)
                num_std_tri = 0
                for ses in os.listdir(sub_path):
                    ses_path = os.path.join(sub_path, ses)
                    ses_path_sp = os.path.join(ses_path, os.listdir(ses_path)[0]) # only one subset
                    for tri in os.listdir(ses_path_sp):
                        tri_path = os.path.join(ses_path_sp, tri)
                        f = pyedflib.EdfReader(tri_path)
                        channels = f.getSignalLabels()
                        unique_channels.update(channels)
                        f.close()
                        if all([ch in channels for ch in li_std_ch_1]):
                            num_std_tri += 1
                        elif all([ch in channels for ch in li_std_ch_2]):
                            num_std_tri += 1
                if num_std_tri >= 1:
                    li_std_sub.append(sub_path)
                                            
    return li_std_sub, unique_channels

# select subjects with all 19 standard channels
li_std_sub, unique_channels = std_sub(root_path, li_std_ch_1, li_std_ch_2)
print(f"Number of subjects with standard channels: ", len(li_std_sub))
print("Unique channels found across all subjects:", unique_channels)
print("Number of unique channels:", len(unique_channels))

Number of subjects with standard channels:  200
Unique channels found across all subjects: {'EEG T2-LE', 'EEG 26-REF', 'EEG F3-REF', 'EEG 31-LE', 'EEG F8-LE', 'EEG PG1-LE', 'DC5-DC', 'DC4-DC', 'EEG C3P-REF', 'EEG C4P-REF', 'EEG LOC-REF', 'EEG EKG-LE', 'EEG OZ-LE', 'EEG A1-LE', 'EEG F4-REF', 'EEG T4-LE', 'EEG O1-REF', 'EEG RLC-LE', 'PHOTIC-REF', 'EEG F8-REF', 'EEG 24-REF', 'EEG T4-REF', 'EEG PZ-REF', 'EEG T1-REF', 'EEG 20-REF', 'EEG T3-REF', 'EEG 30-LE', 'EEG 28-REF', 'EEG 28-LE', 'EEG C4-LE', 'EEG P3-REF', 'EEG 29-REF', 'EEG C4-REF', 'EEG FP1-LE', 'EEG 27-REF', 'EEG A2-REF', 'BURSTS', 'IBI', 'DC7-DC', 'EEG T5-LE', 'EEG 32-LE', 'EEG 22-REF', 'EEG T2-REF', 'EEG FZ-REF', 'EEG 21-REF', 'EEG T6-LE', 'EEG T1-LE', 'EEG A1-REF', 'EEG T5-REF', 'EEG 31-REF', 'EEG 29-LE', 'SUPPR', 'EEG SP1-REF', 'EEG PZ-LE', 'EEG FP2-LE', 'PHOTIC PH', 'EEG F7-REF', 'EEG 27-LE', 'EEG FP2-REF', 'EEG SP2-LE', 'EEG T3-LE', 'EEG 30-REF', 'EEG 25-REF', 'EEG T6-REF', 'EEG SP1-LE', 'EMG-REF', 'EEG O2-LE', 'EEG P4-LE', 'D

## Features

In [6]:
def data_preprocessing(
    eeg_data: np.ndarray,
    ch_names: list,
    source_sample_rate: int,
    target_sample_rate: int = 200,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 45.0,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        eeg_data (np.ndarray): EEG data, shape (T, C).
        ch_names (list): List of channel names.
        source_sample_rate (int): Original sampling frequency of the EEG data.
        target_sample_rate (int): Desired sampling frequency after resampling (default: 200 Hz).
        notch_freq (float): Notch filter frequency (default: 60 Hz).
        l_freq (float): Low cutoff frequency for bandpass filter (default: 0.5 Hz).
        h_freq (float): High cutoff frequency for bandpass filter (default: 40 Hz).
        verbose (bool): Verbose output.

    Returns:
        mne.io.Raw: Preprocessed MNE Raw object.
    """
    # 1. Construct MNE Raw object
    info = mne.create_info(ch_names=ch_names, sfreq=source_sample_rate, ch_types=['eeg'] * len(ch_names))
    raw = mne.io.RawArray(eeg_data.T, info)

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Montage set: 'standard_1020'.")
        
    # 3. Notch Filter at 50 Hz
    raw.notch_filter(freqs=notch_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Notch @ {notch_freq} Hz")

    #  Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    if verbose:
        print(f"✔ Bandpass filter applied ({l_freq}–{h_freq} Hz).")
        
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
            
    # 8) downsample to target sample rate
    if raw.info["sfreq"] != target_sample_rate:
        raw.resample(target_sample_rate, npad="auto", verbose=False)
    if verbose:
        print(f"✔ Resampled to {target_sample_rate} Hz")
        
    return raw
        

def eeg_data(std_edf_path, li_std_ch, preprocess=True):
    signals, signal_headers, _ = highlevel.read_edf(std_edf_path, ch_names=li_std_ch)
    freq = signal_headers[0]['sample_frequency']
    # preprocess the data, data collected in the United States, notch frequency is 60Hz
    # replace channel names with standard channel names
    ch_names = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
    if preprocess:
        raw = data_preprocessing(signals.T, ch_names, freq, notch_freq=60.0, l_freq=0.5, h_freq=45.0, verbose=True)
        data = raw.get_data().T
    else:
        data = signals.T
    return data, freq

In [7]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [8]:
trial_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

# Since each subject has multiple sessions, each session has multiple trials, and many sessions are not even collected in the same year, we will save each trial as a separate npy file.
# For our contrastive learning task, we will use the trial ID as the guide ID for positive pairs for this dataset.
# To avoid overfitting, we will limit the maximum number of timepoints per trial to 1000000.
trial_id = 1
for sub_path in li_std_sub:
    print(sub_path)
    for ses in os.listdir(sub_path):
        ses_path = os.path.join(sub_path, ses)
        ses_path_sp = os.path.join(ses_path, os.listdir(ses_path)[0]) # only one subset
        for tri in os.listdir(ses_path_sp):
            tri_path = os.path.join(ses_path_sp, tri)
            try:
                f = pyedflib.EdfReader(tri_path)
            except Exception:
                continue
            channels = f.getSignalLabels()
            f.close()
            if all([ch in channels for ch in li_std_ch_1]):  # check if all 19 channels exist
                data, original_fs = eeg_data(tri_path, li_std_ch_1, preprocess=False)
            elif all([ch in channels for ch in li_std_ch_2]):
                data, original_fs = eeg_data(tri_path, li_std_ch_2, preprocess=False)
            # length if resampled to TARGET_SAMPLE_RATE, preserve for later min_timepoints check
            T_raw = int(data.shape[0] * TARGET_SAMPLE_RATE / original_fs)  
            print(f"Subject {sub_path} Session {ses} Trial {tri} has length {T_raw} if resampling to {TARGET_SAMPLE_RATE}. The original sampling rate was {original_fs}Hz \n")
            # Take the middle MAX_TIMEPOINT_PER_TRIAL timepoints
            if T_raw > MAX_TIMEPOINT_PER_TRIAL:
                T_raw = MAX_TIMEPOINT_PER_TRIAL
                print(f"Prepare to take the middle {MAX_TIMEPOINT_PER_TRIAL} timepoints.")
            if T_raw < MIN_TIMEPOINT_PER_TRIAL:
                print(f"Trial {tri} has less than {MIN_TIMEPOINT_PER_TRIAL} timepoints, skip this one.")
                continue
            for fs in SAMPLE_RATE_LIST:
                T_res = int(T_raw * fs / TARGET_SAMPLE_RATE)   # now the raw sampling rate is TARGET_SAMPLE_RATE
                # compute number of segments
                n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                trial_segment_counts[trial_id][fs] += n_seg
                N_total += n_seg
                print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                print("----------------------------------------")
            trial_id += 1
            print("----------Trial End----------")
    print("----------------------------------Subject End-------------------------------------\n")
    
    
print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

TUEP//v2.0.0\epilepsy_edf\aaaaaanr
Subject TUEP//v2.0.0\epilepsy_edf\aaaaaanr Session s001_2003_10_14 Trial aaaaaanr_s001_t001.edf has length 242800 if resampling to 200. The original sampling rate was 250.0Hz 

fs=200Hz: T_res=242800, STEP=200, n_seg=1213
----------------------------------------
fs=100Hz: T_res=121400, STEP=200, n_seg=606
----------------------------------------
fs=50Hz: T_res=60700, STEP=200, n_seg=302
----------------------------------------
----------Trial End----------
Subject TUEP//v2.0.0\epilepsy_edf\aaaaaanr Session s002_2009_12_03 Trial aaaaaanr_s002_t000.edf has length 26000 if resampling to 200. The original sampling rate was 250.0Hz 

fs=200Hz: T_res=26000, STEP=200, n_seg=129
----------------------------------------
fs=100Hz: T_res=13000, STEP=200, n_seg=64
----------------------------------------
fs=50Hz: T_res=6500, STEP=200, n_seg=31
----------------------------------------
----------Trial End----------
Subject T

In [9]:
output_root = os.path.join('Processed', sub_folder_path, 'TUEP')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))
print("Created memmap for X with shape:", X_mm.shape)

X path: Processed\L400\TUEP\X.dat
y path: Processed\L400\TUEP\y.dat
Created memmap for X with shape: (3946935, 400, 19)


## PASS 2: Process and store segments into memmap

In [10]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
trial_id = 1  # trial ID starts from 1
session_id = 1  # session ID starts from 1
sub_id = 1  # subject ID starts from 1
for sub_path in li_std_sub:
    print(sub_path)
    sub_label = 0  # use for pre-training, all subjects are labeled as 0
    for ses in os.listdir(sub_path):
        ses_path = os.path.join(sub_path, ses)
        ses_path_sp = os.path.join(ses_path, os.listdir(ses_path)[0]) # only one subset
        for tri in os.listdir(ses_path_sp):
            tri_path = os.path.join(ses_path_sp, tri)
            try:
                f = pyedflib.EdfReader(tri_path)
            except Exception:
                continue
            channels = f.getSignalLabels()
            f.close()
            if all([ch in channels for ch in li_std_ch_1]):  # check if all 19 channels exist
                data, original_fs = eeg_data(tri_path, li_std_ch_1, preprocess=True)
            elif all([ch in channels for ch in li_std_ch_2]):
                data, original_fs = eeg_data(tri_path, li_std_ch_2, preprocess=True)
            print(f"Subject {sub_path} Session {ses} Trial {tri} has shape {data.shape} and sampling rate {TARGET_SAMPLE_RATE}. The original sampling rate was {original_fs}Hz \n")
            # Take the middle MAX_TIMEPOINT_PER_TRIAL timepoints
            if data.shape[0] > MAX_TIMEPOINT_PER_TRIAL:
                start_idx = (data.shape[0] - MAX_TIMEPOINT_PER_TRIAL) // 2
                end_idx = start_idx + MAX_TIMEPOINT_PER_TRIAL
                data = data[start_idx:end_idx, :]
                print(f"Trial {tri} has more than {MAX_TIMEPOINT_PER_TRIAL} timepoints, taking the middle {MAX_TIMEPOINT_PER_TRIAL} timepoints.")
            elif data.shape[0] < MIN_TIMEPOINT_PER_TRIAL:
                print(f"Trial {tri} has less than {MIN_TIMEPOINT_PER_TRIAL} timepoints, skip this one.")
                continue
            T_raw = data.shape[0]
            total_seconds_all += data.shape[0] / TARGET_SAMPLE_RATE
            data = data.astype('float32')  # convert to float32 to save space
            for fs in SAMPLE_RATE_LIST:
                # resample to target fs
                data_resampled = resample_time_series(data, TARGET_SAMPLE_RATE, fs)  # data has already been preprocessed to TARGET_SAMPLE_RATE
                T_res, _ = data_resampled.shape
                print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
    
                # overlapped sliding window with fixed STEP (in timestamps)
                starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                print(f"fs={fs}Hz: segments={len(starts)}")
    
                for s in starts:
                    if cur >= N_total:
                        raise RuntimeError("Exceeded predicted N_total.")
    
                    X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                    y_mm[cur, 0] = float(sub_label)       # label
                    y_mm[cur, 1] = float(trial_id)      # Global trial ID
                    # y_mm[cur, 2] = float(session_id)     # session ID
                    # y_mm[cur, 3] = float(sub_id)       # subject ID
                    y_mm[cur, 2] = float(fs)               # sampling frequency
                    cur += 1
            trial_id += 1
            print("----------Trial End----------")
        session_id += 1
        print("-----------------------Session End-------------------------\n")
    sub_id += 1
    print("----------------------------------Subject End-------------------------------------\n")
    
    
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

TUEP//v2.0.0\epilepsy_edf\aaaaaanr
Creating RawArray with float64 data, n_channels=19, n_times=303500
    Range : 0 ... 303499 =      0.000 ...  1213.996 secs
Ready.
✔ Montage set: 'standard_1020'.
✔ Notch @ 60.0 Hz
✔ Bandpass filter applied (0.5–45.0 Hz).
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 19 components
Fitting ICA took 15.5s.
✔ ICA fitted.
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 3 ICA components
    Projecting back using 19 PCA components
✔ ICA applied. Excluded components: [0, 4, 7]
✔ Resampled to 200 Hz
Subject TUEP//v2.0.0\epilepsy_edf\aaaaaanr Session s001_2003_10_14 Trial aaaaaanr_s001_t001.edf has shape (242800, 19) and sampling rate 200. The original sampling rate was 250.0Hz 

fs=200Hz: resampled shape=(242800, 19)
fs=200Hz: segments=1213
fs=100Hz: resampled shape=(121400, 19)
fs=100Hz: segments=606
fs=50Hz: resampled shape=(60700, 19)
fs=50Hz

## Create JSON file

In [11]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "MONTAGE": "standard_1020",
    "CHANNELS": standard_channels,
    "LABELS": {
        "disease_id": {0: "no epilepsy", 1: "epilepsy" },
        "trial_id": "integer from 1 to number of global trials",
        "sampling_frequency": "integer in SAMPLE_RATE_LIST"
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\L400\TUEP\meta.json


## Load and check the processed data

In [13]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

total_samples = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject
    total_samples += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total samples:", total_samples)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 3946935
T: 400
C: 19
SAMPLE_RATE_LIST: [200, 100, 50]
MONTAGE: standard_1020
CHANNELS: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
LABELS: {'disease_id': {'0': 'no epilepsy', '1': 'epilepsy'}, 'trial_id': 'integer from 1 to number of global trials', 'sampling_frequency': 'integer in SAMPLE_RATE_LIST'}
-------------------------------------
Subject ID 001: X shape=(2121, 400, 19), y shape=(2121, 3)
Subject ID 002: X shape=(224, 400, 19), y shape=(224, 3)
Subject ID 003: X shape=(2188, 400, 19), y shape=(2188, 3)
Subject ID 004: X shape=(2315, 400, 19), y shape=(2315, 3)
Subject ID 005: X shape=(2220, 400, 19), y shape=(2220, 3)
Subject ID 006: X shape=(193, 400, 19), y shape=(193, 3)
Subject ID 007: X shape=(326, 400, 19), y shape=(326, 3)
Subject ID 008: X shape=(1048, 400, 19), y shape=(1048, 3)
Subject ID 009: X shape=(1048, 400, 19), y shape=(1048, 3)
Subject ID 010: X shape=(522, 400, 19), y shape=(522